# 可选实验：Neurons 与 Layers

本实验将探索 neuron／unit（神经元／单元）和 layer（层）的内部计算，并与 Course 1 中的 linear regression、logistic regression 模型进行对照。我们还将介绍 TensorFlow，展示这些模型在该框架中的实现。

<figure>
  <img src="./images/C2_W1_NeuronsAndLayers.png" alt="Neurons 与 layers 示意图" style="width:540px;max-width:100%;height:auto;">
</figure>


## 依赖库

**TensorFlow 与 Keras**

TensorFlow 是 Google 开发的 Machine Learning 库。2019 年，Google 在发布 TensorFlow 2.0 时整合了 Keras。Keras 最初由 François Chollet 独立开发，为构建模型提供了简洁、以 layer 为中心的接口。本课程使用 TensorFlow 中的 Keras 接口。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras import Sequential
from tensorflow.keras.losses import MeanSquaredError, BinaryCrossentropy
from tensorflow.keras.activations import sigmoid
from lab_utils_common import dlc
from lab_neurons_utils import plt_prob_1d, sigmoidnp, plt_linear, plt_logistic
plt.style.use('./deeplearning.mplstyle')
import logging
logging.getLogger("tensorflow").setLevel(logging.ERROR)
tf.autograph.set_verbosity(0)

## 不使用非线性 Activation 的 Neuron：Linear Regression


### Linear Regression 数据集

我们使用 Course 1 中通过 linear regression 预测房价的示例。


In [ ]:
X_train = np.array([[1.0], [2.0]], dtype=np.float32)           #(size in 1000 square feet)
Y_train = np.array([[300.0], [500.0]], dtype=np.float32)       #(price in 1000s of dollars)

fig, ax = plt.subplots(1,1)
ax.scatter(X_train, Y_train, marker='x', c='r', label="Data Points")
ax.legend( fontsize='xx-large')
ax.set_ylabel('Price (in 1000s of dollars)', fontsize='xx-large')
ax.set_xlabel('Size (1000 sqft)', fontsize='xx-large')
plt.show()

### Linear Regression 模型

不使用非线性 activation function 的 neuron，其计算与 Course 1 的 linear regression 相同：

$$
f_{\mathbf{w},b}(\mathbf{x}^{(i)})=\mathbf{w}\cdot\mathbf{x}^{(i)}+b \tag{1}
$$


下面定义一个只包含一个 neuron 的 layer，并与熟悉的 linear regression 函数比较。


In [ ]:
linear_layer = tf.keras.layers.Dense(units=1, activation = 'linear', )

下面查看 weight。


In [ ]:
linear_layer.get_weights()

此时还没有 weight，因为参数尚未创建。下面用 `X_train` 中的一个样本调用 layer，触发参数的创建。注意，输入必须是二维的，因此需要先调整 shape。


In [ ]:
a1 = linear_layer(X_train[0].reshape(1,1))
print(a1)

结果是 shape 为 `(1, 1)` 的 tensor（张量），其中只有一个元素。Tensor 是框架中表示 array 的数据结构。

下面查看 weight 和 bias：weight 随机初始化为较小的数值，bias 默认初始化为 0。


In [ ]:
w, b= linear_layer.get_weights()
print(f"w = {w}, b={b}")

只有一个输入 feature 的 linear regression 模型包含一个 weight 和一个 bias，与上面的 `linear_layer` 一致。

当前 weight 是随机值，下面将参数设为已知值。


In [ ]:
set_w = np.array([[200]])
set_b = np.array([100])

# set_weights takes a list of numpy arrays
linear_layer.set_weights([set_w, set_b])
print(linear_layer.get_weights())

下面比较公式 (1) 与 layer 的输出。


In [ ]:
a1 = linear_layer(X_train[0].reshape(1,1))
print(a1)
alin = np.dot(set_w,X_train[0].reshape(1,1)) + set_b
print(alin)

两者得到的值相同！现在可以使用这个 linear layer 对训练数据进行预测。


In [ ]:
prediction_tf = linear_layer(X_train)
prediction_np = np.dot( X_train, set_w) + set_b

In [ ]:
plt_linear(X_train, Y_train, prediction_tf, prediction_np)

## 使用 Sigmoid Activation 的 Neuron

使用 sigmoid activation 的 neuron，其计算与 Course 1 的 logistic regression 相同：

$$
f_{\mathbf{w},b}(\mathbf{x}^{(i)})=g(\mathbf{w}\cdot\mathbf{x}^{(i)}+b) \tag{2}
$$

其中 $g(z)=\operatorname{sigmoid}(z)$。

下面将 $w$ 和 $b$ 设为已知值，检查模型输出。


### Logistic Regression 数据集

我们使用 Course 1 中的 logistic regression 示例。


In [ ]:
X_train = np.array([0., 1, 2, 3, 4, 5], dtype=np.float32).reshape(-1,1)  # 2-D Matrix
Y_train = np.array([0,  0, 0, 1, 1, 1], dtype=np.float32).reshape(-1,1)  # 2-D Matrix

In [ ]:
pos = Y_train == 1
neg = Y_train == 0

fig,ax = plt.subplots(1,1,figsize=(4,3))
ax.scatter(X_train[pos], Y_train[pos], marker='x', s=80, c = 'red', label="y=1")
ax.scatter(X_train[neg], Y_train[neg], marker='o', s=100, label="y=0", facecolors='none', 
              edgecolors=dlc["dlblue"],lw=3)

ax.set_ylim(-0.08,1.1)
ax.set_ylabel('y', fontsize=12)
ax.set_xlabel('x', fontsize=12)
ax.set_title('one variable plot')
ax.legend(fontsize=12)
plt.show()

### Logistic Neuron

加入 sigmoid activation 后，就得到按照公式 (2) 计算的 logistic neuron。

这里将创建一个包含 logistic layer 的 TensorFlow model，展示另一种构建方式。TensorFlow 通常用于构建多层模型，[Sequential](https://keras.io/guides/sequential_model/) 是一种方便的构建接口。


In [ ]:
model = Sequential(
    [
        tf.keras.layers.Dense(1, input_dim=1,  activation = 'sigmoid', name='L1')
    ]
)

`model.summary()` 显示模型中的 layer 及参数数量。这个模型只有一个 layer，其中只有一个 unit；该 unit 包含 $w$ 和 $b$ 两个参数。


In [ ]:
model.summary()

In [ ]:
logistic_layer = model.get_layer('L1')
w,b = logistic_layer.get_weights()
print(w,b)
print(w.shape,b.shape)

下面将 weight 和 bias 设为已知值。


In [ ]:
set_w = np.array([[2]])
set_b = np.array([-4.5])
# set_weights takes a list of numpy arrays
logistic_layer.set_weights([set_w, set_b])
print(logistic_layer.get_weights())

下面比较公式 (2) 与 layer 的输出。


In [ ]:
a1 = model.predict(X_train[0].reshape(1,1))
print(a1)
alog = sigmoidnp(np.dot(set_w,X_train[0].reshape(1,1)) + set_b)
print(alog)

两者得到的值相同！现在可以分别用 logistic layer 和 NumPy 模型预测训练数据。


In [ ]:
plt_logistic(X_train, Y_train, model, set_w, set_b, pos, neg)

上图的颜色深浅表示 sigmoid 的输出，其值位于 0 与 1 之间。


## 恭喜完成实验！

你构建了一个简单的 neural network（神经网络），并探索了 neuron 与 Course 1 中 linear regression、logistic regression 的相似之处。
